# 🧠 EX58: การส่งออกโมเดล (Model Export & Optimization)

| รูปแบบ | ประโยชน์ | API |
|--------|---------|-----|
| **ONNX** | ทุก platform | `format="onnx"` |
| **TensorRT** | GPU throughput สูงสุด | `format="engine"` |
| **TFLite** | Mobile/Edge TPU | `format="tflite"` |
| **OpenVINO** | Intel CPU/NPU | `format="openvino"` |

- **half=True** → FP16: throughput 2× บน Tensor Core, ขนาดลด ~50%
- **int8=True** → INT8: บีบอัด 4×, ต้องใช้ภาพ calibration

> [!WARNING]
> TensorRT `.engine` ขึ้นตรงกับ hardware — compile บนเครื่องที่จะ deploy เสมอ

## 🔗 ลิงก์
- [[EX57_Bounding_Box_Parsing_TH]] | [[YOLO_Learning_Plan]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, os, time
import numpy as np, torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
from solution import export_yolo_model
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] ใช้อุปกรณ์: {device}")

print("\n--- เริ่มการตรวจสอบ ---")
print("Export yolo11n.pt → ONNX...")
onnx_path = export_yolo_model("yolo11n.pt", "onnx")
pt_mb   = os.path.getsize("yolo11n.pt") / 1024**2
onnx_mb = os.path.getsize(onnx_path) / 1024**2
print(f"  .pt={pt_mb:.2f} MB | .onnx={onnx_mb:.2f} MB")

print("\nวัด latency (10 warmup + 20 runs)...")
dummy = np.zeros((480, 640, 3), dtype=np.uint8)
model_pt = YOLO("yolo11n.pt"); model_onnx = YOLO(onnx_path)
for _ in range(10):
    model_pt.predict(source=dummy, verbose=False)
    model_onnx.predict(source=dummy, verbose=False)

pt_t, onnx_t = [], []
for _ in range(20):
    t=time.perf_counter(); model_pt.predict(source=dummy, verbose=False);   pt_t.append((time.perf_counter()-t)*1000)
    t=time.perf_counter(); model_onnx.predict(source=dummy, verbose=False); onnx_t.append((time.perf_counter()-t)*1000)

print(f"  PyTorch: {np.mean(pt_t):.1f} ± {np.std(pt_t):.1f} ms")
print(f"  ONNX:    {np.mean(onnx_t):.1f} ± {np.std(onnx_t):.1f} ms")
print("--- สิ้นสุดการตรวจสอบ ---")

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].bar([".pt",".onnx"],[pt_mb,onnx_mb],color=["#e74c3c","#3498db"])
axes[0].set_ylabel("ขนาด (MB)"); axes[0].set_title("ขนาดไฟล์")
axes[1].boxplot([pt_t,onnx_t],labels=["PyTorch","ONNX"])
axes[1].set_ylabel("Latency (ms)"); axes[1].set_title("Benchmark (20 runs)")
plt.tight_layout(); plt.show()

del model_pt, model_onnx
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
